# PlantCLEF 2026 — soumission

Notebook de soumission pour Kaggle. La structure est celle de `submission-method.md` : les cellules 1 à 4, 6 et 7 sont fixes. Seule `predict_species` (cellule 5) change d'un run à l'autre.

**Ce run** découpe chaque quadrat en grille régulière 4×4. Une tuile n'est envoyée au modèle que si au moins 1 % de ses pixels (1 sur 8) ont une couleur de fleur : bleu, violet, rose, magenta, rouge, orange vif, jaune saturé, ou blanc franc. Les tuiles surtout vertes, marron ou grises sont laissées de côté. Le score d'une espèce est le maximum de ses probabilités softmax sur les tuiles retenues. La fonction renvoie les espèces de score au moins `0.1`, par score décroissant. Aucune tuile retenue, ou aucune espèce au-dessus du seuil, donne une liste vide.


In [ ]:
import os, csv
import numpy as np
import pandas as pd
import torch
import timm
from PIL import Image
from tqdm import tqdm


In [ ]:
C = "/kaggle/input/competitions/plantclef-2026"
TEST_CSV    = f"{C}/PlantCLEF2025_test.csv"
SPECIES_CSV = f"{C}/species_ids.csv"
IMG_DIR     = f"{C}/PlantCLEF2025_test_images/PlantCLEF2025_test_images"
MODEL_CKPT  = "/kaggle/input/models/juliostat/dinov2_patch14_reg4_onlyclassifier_then_all/pytorch/default/3/model_best.pth.tar"

ID_COL  = "quadrat_id"   # vérifié : colonnes du CSV de test = quadrat_id, author, date, license
IMG_EXT = ".jpg"         # vérifié : les 2 105 images sont en .jpg, noms = quadrat_id + extension

def image_path(quadrat_id: str) -> str:
    return f"{IMG_DIR}/{quadrat_id}{IMG_EXT}"

print("GPU :", torch.cuda.is_available())


In [ ]:
df_species = pd.read_csv(SPECIES_CSV)
class_map = df_species["species_id"].to_dict()   # indice de sortie -> species_id

# Contrôle : d'après la page Zenodo des modèles, la sortie 2 correspond à l'espèce 1355870.
assert class_map[2] == 1355870, "species_ids.csv ne suit pas l'ordre des sorties du modèle (class_mapping.txt n'est pas dans les entrées)"


In [ ]:
device = torch.device("cuda")
model = timm.create_model(
    "vit_base_patch14_reg4_dinov2.lvd142m",
    pretrained=False,                 # pas de téléchargement : les poids viennent de MODEL_CKPT
    num_classes=len(df_species),
    checkpoint_path=MODEL_CKPT,
)
model = model.to(device).eval()
data_config = timm.data.resolve_model_data_config(model)


In [ ]:
from matplotlib.colors import rgb_to_hsv

# Grille régulière : seules les tuiles assez « fleur » passent dans le modèle, puis max des softmax.
GRID_N = 4
FLOWER_FRAC_MIN = 0.01  # part minimale de pixels « fleur » pour traiter une tuile
SCORE_MIN = 0.1         # probabilité softmax minimale pour retenir une espèce
KEEP_WHITE = True       # False si un cadre ou des reflets blancs sont pris pour des fleurs
SAMPLE_STEP = 8         # 1 pixel sur 8 pour estimer la fraction colorée

_RESAMPLE = {
    "nearest": Image.Resampling.NEAREST,
    "bilinear": Image.Resampling.BILINEAR,
    "bicubic": Image.Resampling.BICUBIC,
}


def flower_fraction(tiles, keep_white=True, step=8):
    """tiles : tenseur (N, 3, H, W) en [0, 1]. Fraction de pixels « fleur » par tuile.
    On sous-échantillonne (1 pixel sur `step`)."""
    sample = tiles[:, :, ::step, ::step].permute(0, 2, 3, 1).contiguous().numpy()
    hsv = rgb_to_hsv(sample)
    h, s, v = hsv[..., 0] * 360, hsv[..., 1], hsv[..., 2]

    blue_pink = (h >= 170) & (h < 345) & (s >= 0.25) & (v >= 0.25)    # bleu, violet, rose, magenta
    red_orange = ((h >= 345) | (h < 50)) & (s >= 0.35) & (v >= 0.75)  # rouge, orange vif (le marron est plus sombre)
    yellow = (h >= 50) & (h < 70) & (s >= 0.45) & (v >= 0.6)          # jaune saturé
    mask = blue_pink | red_orange | yellow
    if keep_white:
        mask |= (s < 0.15) & (v >= 0.9)                                # blanc franc (pâquerettes)
    return mask.mean(axis=(1, 2))


def make_tiles(img, n, size, resample):
    """n×n tuiles redimensionnées à `size` (largeur, hauteur), en [0, 1], forme (n*n, 3, H, W)."""
    width, height = img.size
    tiles = []
    for row in range(n):
        for col in range(n):
            box = (
                round(col * width / n),
                round(row * height / n),
                round((col + 1) * width / n),
                round((row + 1) * height / n),
            )
            tiles.append(np.asarray(img.crop(box).resize(size, resample)))
    return torch.from_numpy(np.stack(tiles)).permute(0, 3, 1, 2).float() / 255


def species_from_scores(best) -> list[int]:
    """Espèces de score >= SCORE_MIN, par score décroissant, sans doublon."""
    order = np.argsort(best)[::-1]
    species = []
    seen = set()
    for index in order:
        if best[index] < SCORE_MIN:
            break
        species_id = int(class_map[int(index)])
        if species_id not in seen:
            seen.add(species_id)
            species.append(species_id)
    return species


def predict_species(quadrat_id: str) -> list[int]:
    """Renvoie les espèces retenues pour un quadrat du test. Le modèle est déjà chargé."""
    path = image_path(quadrat_id)
    if not os.path.isfile(path):
        raise FileNotFoundError(path)

    image = Image.open(path).convert("RGB")
    _channels, out_h, out_w = data_config["input_size"]
    resample = _RESAMPLE[data_config["interpolation"]]
    tiles = make_tiles(image, GRID_N, (out_w, out_h), resample)

    keep = flower_fraction(tiles, KEEP_WHITE, SAMPLE_STEP) >= FLOWER_FRAC_MIN
    if not keep.any():
        return []

    mean = np.asarray(data_config["mean"], dtype=np.float32)
    std = np.asarray(data_config["std"], dtype=np.float32)
    pixels = tiles[torch.from_numpy(np.ascontiguousarray(keep))].permute(0, 2, 3, 1).numpy()
    pixels = (pixels - mean) / std
    batch = torch.from_numpy(np.ascontiguousarray(pixels)).permute(0, 3, 1, 2).to(device)

    with torch.no_grad(), torch.autocast("cuda"):
        best = torch.softmax(model(batch).float(), dim=1).max(dim=0).values.cpu().numpy()
    return species_from_scores(best)


In [ ]:
def create_submission_file(out_path: str = "/kaggle/working/submission.csv", predict=None) -> pd.DataFrame:
    """Parcourt PlantCLEF2025_test.csv dans l'ordre, appelle predict_species, écrit le CSV."""
    if predict is None:
        predict = predict_species

    ids = pd.read_csv(TEST_CSV, sep=";")[ID_COL].astype(str).tolist()   # ce CSV est séparé par des ;

    rows = []
    for qid in tqdm(ids):
        species = list(dict.fromkeys(int(s) for s in predict(qid)))   # int Python + sans doublons
        rows.append(str(species))

    df_sub = pd.DataFrame({"quadrat_id": ids, "species_ids": rows})
    df_sub.to_csv(out_path, index=False, quoting=csv.QUOTE_ALL)
    return df_sub


In [ ]:
df_sub = create_submission_file()
print(df_sub.shape)
print(df_sub.head())
print("quadrats sans espèce :", (df_sub["species_ids"] == "[]").sum())
